# 07b - TPI Validation

## Objective

This notebook validates the implementation of **Text Preserved Information (TPI)**
before applying it to the interview anonymization corpus.

The validation follows the same logic used for the previous content-preservation
notebooks:

1. **Sanity checks** for the TPI implementation.
2. **Controlled benchmark** with known transformations.
3. **Expected-direction tests**: more affected information should produce lower TPI.
4. **Replacement, deletion, generalization, and multiple-modification tests**.
5. **Term/token alignment checks**, including punctuation handling.
6. **S-TA vs N-TA consistency check** on a short text.

TPI measures the proportion of the original text's estimated information content
that is **not associated with affected original terms**.

$$
IC(t) = -\log P(t)
$$

$$
TIC(W_o) = \sum_{t \in W_o} IC(t)
$$

$$
TPI = 1 - \frac{\sum_{t \in M} IC(t)}{TIC(W_o)}
$$

where $M$ is the set of terms affected by anonymization.

> **Important:** TPI does not decide whether a replacement is semantically good.
> It quantifies the information content associated with the affected original
> material. Semantic similarity and NLI remain separate evaluation signals.


## 1. Imports and model

We use the same implementation as the main TPI notebook:

- `bert-base-uncased`
- WordPiece tokenization
- N-Term Alternating (N-TA)
- $N=6$ as the default strategy

The model is only used to estimate contextual probabilities.


In [51]:
import math
import re
import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForMaskedLM

MODEL_NAME = "bert-base-uncased"
N = 6

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("Model:", MODEL_NAME)
print("Device:", device)
print("N-TA N:", N)


Loading weights: 100%|█████████████████████| 202/202 [00:00<00:00, 8647.87it/s]
[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model: bert-base-uncased
Device: cpu
N-TA N: 6


## 2. Term extraction and WordPiece alignment

The previous TPI notebook used a regex that excludes punctuation from terms.

This is important for validation: for example, in

> `I moved to Geneva in 2022.`

the final term must be `2022`, not `2022.`.

This notebook therefore keeps the same punctuation-excluding definition and
uses the **same function consistently when specifying affected terms**.


In [52]:
def whitespace_terms(text):
    """Return alphanumeric linguistic terms with character spans, excluding punctuation."""
    return [
        {
            "term": m.group(),
            "start": m.start(),
            "end": m.end(),
        }
        for m in re.finditer(r"[A-Za-z0-9]+(?:['-][A-Za-z0-9]+)*", text)
    ]


def term_subtokens(term):
    """Return BERT WordPiece tokens for a term."""
    return tokenizer.tokenize(term)


alignment_example = "I moved to Geneva in 2022."

terms = whitespace_terms(alignment_example)

alignment_table = pd.DataFrame([
    {
        "index": i,
        "term": x["term"],
        "start": x["start"],
        "end": x["end"],
        "subtokens": term_subtokens(x["term"]),
    }
    for i, x in enumerate(terms)
])

display(alignment_table)

assert "2022" in alignment_table["term"].tolist()
assert "2022." not in alignment_table["term"].tolist()

print("✓ Punctuation alignment check passed.")


,index,term,start,end,subtokens
0,0,I,0,1,[i]
1,1,moved,2,7,[moved]
2,2,to,8,10,[to]
3,3,Geneva,11,17,[geneva]
4,4,in,18,20,[in]
5,5,2022,21,25,"[202, ##2]"


✓ Punctuation alignment check passed.


In [53]:
def build_masked_text(text, target_indices):
    """Replace selected terms by the required number of [MASK] tokens."""
    terms = whitespace_terms(text)
    target_indices = set(target_indices)

    pieces = []
    cursor = 0
    mask_lengths = {}

    for i, item in enumerate(terms):
        pieces.append(text[cursor:item["start"]])

        if i in target_indices:
            subtokens = term_subtokens(item["term"])
            pieces.append(" ".join([tokenizer.mask_token] * len(subtokens)))
            mask_lengths[i] = len(subtokens)
        else:
            pieces.append(text[item["start"]:item["end"]])

        cursor = item["end"]

    pieces.append(text[cursor:])

    return "".join(pieces), mask_lengths


def masked_term_probabilities(text, target_indices):
    """Estimate contextual probabilities for several masked terms in one pass."""
    terms = whitespace_terms(text)
    masked_text, mask_lengths = build_masked_text(text, target_indices)

    inputs = tokenizer(masked_text, return_tensors="pt").to(device)

    with torch.no_grad():
        logits = model(**inputs).logits[0]
        probabilities = torch.softmax(logits, dim=-1)

    input_ids = inputs["input_ids"][0]
    mask_positions = [
        i for i, token_id in enumerate(input_ids)
        if int(token_id) == tokenizer.mask_token_id
    ]

    results = {}
    cursor = 0

    for idx in sorted(target_indices):
        term = terms[idx]["term"]
        original_subtokens = term_subtokens(term)
        n_subtokens = len(original_subtokens)
        positions = mask_positions[cursor:cursor + n_subtokens]

        if len(positions) != n_subtokens:
            raise RuntimeError(
                f"Could not align masks for term '{term}'."
            )

        token_probs = []
        for position, subtoken in zip(positions, original_subtokens):
            token_id = tokenizer.convert_tokens_to_ids(subtoken)
            token_probs.append(
                float(probabilities[position, token_id].item())
            )

        results[idx] = {
            "term": term,
            "subtokens": original_subtokens,
            "subtoken_probabilities": token_probs,
            "probability": min(token_probs),
        }

        cursor += n_subtokens

    return results


def information_content(probability):
    """IC(t) = -log P(t)."""
    probability = max(float(probability), 1e-24)
    return -math.log(probability)


def estimate_all_term_ic(text, strategy="N-TA", N=6):
    """Estimate IC for every term using S-TA or N-TA."""
    terms = whitespace_terms(text)
    n_terms = len(terms)

    if n_terms == 0:
        return pd.DataFrame(columns=["index", "term", "probability", "IC"])

    if strategy == "S-TA":
        groups = [[i] for i in range(n_terms)]
    elif strategy == "N-TA":
        groups = [
            list(range(offset, n_terms, N))
            for offset in range(N)
            if offset < n_terms
        ]
    else:
        raise ValueError("strategy must be 'S-TA' or 'N-TA'.")

    records = []

    for group in groups:
        group_results = masked_term_probabilities(text, group)

        for idx in group:
            r = group_results[idx]
            records.append({
                "index": idx,
                "term": r["term"],
                "probability": r["probability"],
                "IC": information_content(r["probability"]),
            })

    return pd.DataFrame(records).sort_values("index").reset_index(drop=True)


def total_information_content(ic_table):
    return float(ic_table["IC"].sum())


def information_loss_from_spans(ic_table, affected_term_indices):
    """Return affected terms and their total information content."""
    affected_term_indices = set(affected_term_indices)

    affected = ic_table[
        ic_table["index"].isin(affected_term_indices)
    ].copy()

    return affected, float(affected["IC"].sum())


def tpi_from_spans(ic_table, affected_term_indices):
    """TPI = 1 - affected information content / original TIC."""
    tic = total_information_content(ic_table)

    affected, lost_information = information_loss_from_spans(
        ic_table,
        affected_term_indices
    )

    if tic <= 0:
        raise ValueError("TIC must be strictly positive.")

    tpi = 1.0 - lost_information / tic

    return {
        "TIC_original": tic,
        "information_lost": lost_information,
        "TPI": max(0.0, min(1.0, tpi)),
        "affected_terms": affected,
    }


## 3. Helper for selecting affected terms

Affected spans are defined **on the original text**.

This mirrors the intended use of TPI in the anonymization pipeline:
the anonymization/alignment stage identifies which original spans were
modified or removed; TPI then weights those spans by their information content.


In [54]:
def term_indices(text, selected_terms):
    """Return indices of terms whose normalized form is in selected_terms."""
    selected_terms = {str(x).lower() for x in selected_terms}
    terms = whitespace_terms(text)

    return [
        i for i, item in enumerate(terms)
        if item["term"].lower() in selected_terms
    ]


def run_tpi_case(name, original, anonymized, affected_terms, ic_cache=None):
    """Run one controlled TPI benchmark case."""
    ic_table = (
        ic_cache
        if ic_cache is not None
        else estimate_all_term_ic(original, strategy="N-TA", N=N)
    )

    affected_indices = term_indices(original, affected_terms)
    result = tpi_from_spans(ic_table, affected_indices)

    return {
        "case": name,
        "original": original,
        "anonymized": anonymized,
        "affected_terms": ", ".join(
            ic_table.loc[
                ic_table["index"].isin(affected_indices), "term"
            ].tolist()
        ),
        "information_lost": result["information_lost"],
        "TIC": result["TIC_original"],
        "TPI": result["TPI"],
    }


## 4. Sanity checks

These are the basic implementation checks moved out of the main TPI notebook.

### Check 1 — no affected information

If no original term is affected:

$$
TPI = 1
$$

### Check 2 — all information affected

If every original term is affected:

$$
TPI = 0
$$

These checks validate the basic arithmetic independently of the anonymization
examples.


In [55]:
sanity_text = "I live in Geneva."

ic_sanity = estimate_all_term_ic(
    sanity_text,
    strategy="N-TA",
    N=N
)

tic = total_information_content(ic_sanity)

print("TIC:", tic)
assert tic > 0, "TIC must be strictly positive."

# No affected information
no_affected_result = tpi_from_spans(
    ic_sanity,
    []
)

print("TPI with no affected terms:", no_affected_result["TPI"])

assert np.isclose(
    no_affected_result["TPI"],
    1.0
), "TPI should be 1 when no information is affected."

# All information affected
all_indices = ic_sanity["index"].tolist()

all_affected_result = tpi_from_spans(
    ic_sanity,
    all_indices
)

print("TPI with all terms affected:", all_affected_result["TPI"])

assert np.isclose(
    all_affected_result["TPI"],
    0.0
), "TPI should be 0 when all information is affected."

print("✓ All sanity checks passed.")


TIC: 12.57769510992999
TPI with no affected terms: 1.0
TPI with all terms affected: 0.0
✓ All sanity checks passed.


## 5. Controlled benchmark

We now test the implementation on transformations that represent the main
types of information change expected in anonymization:

- replacement;
- generalization;
- deletion;
- partial deletion;
- multiple modifications.

The anonymized text itself does not enter the TPI formula. It is included
to document the transformation and to make the affected original spans
explicit.


In [56]:
benchmark_cases = [
    {
        "name": "Replacement: location",
        "original": "I live in Lausanne.",
        "anonymized": "I live in Geneva.",
        "affected_terms": ["Lausanne"],
    },
    {
        "name": "Deletion: temporal information",
        "original": "I live in Geneva since 2022.",
        "anonymized": "I live in Geneva.",
        "affected_terms": ["since", "2022"],
    },
    {
        "name": "Deletion: temporal information + Replacement: location",
        "original": "I live in Lausanne since 2022.",
        "anonymized": "I live in Geneva.",
        "affected_terms": ["Lausanne", "since", "2022"],
    },
    {
        "name": "Generalization + deletion",
        "original": "I moved to Geneva in 2022.",
        "anonymized": "I moved to Switzerland.",
        "affected_terms": ["Geneva", "in", "2022"],
    },
    {
        "name": "Replacement/generalization of phrase",
        "original": "I have two brothers.",
        "anonymized": "I have siblings.",
        "affected_terms": ["two", "brothers"],
    },
]

benchmark_rows = []

for case in benchmark_cases:
    row = run_tpi_case(
        name=case["name"],
        original=case["original"],
        anonymized=case["anonymized"],
        affected_terms=case["affected_terms"],
    )
    benchmark_rows.append(row)

benchmark_df = pd.DataFrame(benchmark_rows)

display(
    benchmark_df[
        [
            "case",
            "original",
            "anonymized",
            "affected_terms",
            "information_lost",
            "TIC",
            "TPI",
        ]
    ]
)


,case,original,anonymized,affected_terms,information_lost,TIC,TPI
0,Replacement: location,I live in Lausanne.,I live in Geneva.,Lausanne,8.983905,15.162804,0.407504
1,Deletion: temporal information,I live in Geneva since 2022.,I live in Geneva.,"since, 2022",16.326649,27.483469,0.405947
2,Deletion: temporal information + Replacement: ...,I live in Lausanne since 2022.,I live in Geneva.,"Lausanne, since, 2022",22.245161,28.025356,0.206249
3,Generalization + deletion,I moved to Geneva in 2022.,I moved to Switzerland.,"Geneva, in, 2022",19.552650,28.963484,0.324921
4,Replacement/generalization of phrase,I have two brothers.,I have siblings.,"two, brothers",3.444348,7.066844,0.512605


## 6. Expected-direction tests

The goal here is not to assign a human ground-truth TPI value.

Instead, we test the basic expected behaviour of an information-weighted
measure:

- no information affected → highest possible TPI;
- affecting an additional term → TPI cannot increase;
- affecting all terms → TPI reaches zero.

For a fixed original text, if $M_1 \subseteq M_2$, then:

$$
TPI(M_2) \leq TPI(M_1)
$$

because the information assigned to affected terms is non-negative.


In [57]:
direction_text = "I moved to Geneva in 2022 and I have two brothers."

direction_ic = estimate_all_term_ic(
    direction_text,
    strategy="N-TA",
    N=N
)

idx_geneva = term_indices(direction_text, ["Geneva"])
idx_geneva_year = term_indices(direction_text, ["Geneva", "2022"])
idx_all = direction_ic["index"].tolist()

tpi_none = tpi_from_spans(direction_ic, [])["TPI"]
tpi_geneva = tpi_from_spans(direction_ic, idx_geneva)["TPI"]
tpi_geneva_year = tpi_from_spans(direction_ic, idx_geneva_year)["TPI"]
tpi_all = tpi_from_spans(direction_ic, idx_all)["TPI"]

direction_results = pd.DataFrame({
    "affected_information": [
        "none",
        "Geneva",
        "Geneva + 2022",
        "all terms",
    ],
    "TPI": [
        tpi_none,
        tpi_geneva,
        tpi_geneva_year,
        tpi_all,
    ],
})

display(direction_results)

assert np.isclose(tpi_none, 1.0)
assert tpi_geneva < tpi_none
assert tpi_geneva_year <= tpi_geneva
assert np.isclose(tpi_all, 0.0)

print("✓ Expected-direction tests passed.")


,affected_information,TPI
0,none,1.000000
1,Geneva,0.801788
2,Geneva + 2022,0.375203
3,all terms,0.000000


✓ Expected-direction tests passed.


## 7. Information weighting

TPI is not a simple percentage of words changed.

Two transformations affecting the same number of terms can produce different
TPI values because the terms can have different contextual information
contents.

We therefore inspect the IC values used by the benchmark.


In [58]:
display(
    direction_ic.sort_values("IC", ascending=False)
    [["index", "term", "probability", "IC"]]
)

print(
    "Total information content:",
    total_information_content(direction_ic)
)


,index,term,probability,IC
5,5,2022,1.679999e-07,15.599302
3,3,Geneva,7.114778e-04,7.248166
10,10,brothers,2.228662e-03,6.106354
6,6,and,3.425890e-03,5.676394
9,9,two,3.769791e-01,0.975565
0,0,I,7.041229e-01,0.350802
1,1,moved,7.564459e-01,0.279124
7,7,I,8.232490e-01,0.194497
8,8,have,9.194420e-01,0.083988
4,4,in,9.587823e-01,0.042091


Total information content: 36.56783434739532


## 8. Deletion vs replacement

For TPI, a replacement and a deletion are both represented through the
**affected original span**.

Example:

> `I live in Lausanne.` → `I live in Geneva.`

The original term `Lausanne` is affected.

Example:

> `I live in Geneva since 2022.` → `I live in Geneva.`

The original terms `since` and `2022` are affected.

TPI therefore measures the information content associated with the original
material that is no longer preserved, regardless of whether the anonymizer
replaces it or deletes it.


## 9. Multiple modifications

Real anonymization can modify several speaker-specific elements in the same
utterance.

We therefore verify that TPI accumulates information loss across multiple
affected spans.


In [59]:
multi_text = (
    "I moved from Brussels to Geneva in 2022 "
    "and I have two brothers."
)

multi_ic = estimate_all_term_ic(
    multi_text,
    strategy="N-TA",
    N=N
)

cases = [
    ("No changes", []),
    ("Location only", ["Brussels", "Geneva"]),
    ("Location + year", ["Brussels", "Geneva", "2022"]),
    ("All sensitive details", ["Brussels", "Geneva", "2022", "two", "brothers"]),
]

multi_rows = []

for name, affected in cases:
    result = tpi_from_spans(
        multi_ic,
        term_indices(multi_text, affected)
    )

    multi_rows.append({
        "case": name,
        "affected_terms": ", ".join(
            result["affected_terms"]["term"].tolist()
        ),
        "information_lost": result["information_lost"],
        "TPI": result["TPI"],
    })

multi_df = pd.DataFrame(multi_rows)
display(multi_df)

assert np.isclose(multi_df.loc[0, "TPI"], 1.0)
assert multi_df.loc[1, "TPI"] < multi_df.loc[0, "TPI"]
assert multi_df.loc[2, "TPI"] <= multi_df.loc[1, "TPI"]
assert multi_df.loc[3, "TPI"] <= multi_df.loc[2, "TPI"]

print("✓ Multiple-modification tests passed.")


,case,affected_terms,information_lost,TPI
0,No changes,,0.000000,1.000000
1,Location only,"Brussels, Geneva",9.766476,0.742204
2,Location + year,"Brussels, Geneva, 2022",24.221642,0.360646
3,All sensitive details,"Brussels, Geneva, 2022, two, brothers",31.579881,0.166418


✓ Multiple-modification tests passed.


## 10. S-TA vs N-TA consistency

The main notebook uses N-TA with $N=6$ for efficiency.

As a validation step, we compare N-TA against S-TA on a short sentence.

This is **not** a requirement that the two methods produce exactly identical
values: they use different masking contexts.

The useful check is that:

1. both methods return one IC value per term;
2. probabilities and IC values are finite and positive;
3. the resulting TIC values are in a plausible range;
4. the TPI sanity properties hold for both methods.


In [60]:
comparison_text = "I moved to Geneva from Brussels in 2022."

ic_sta = estimate_all_term_ic(
    comparison_text,
    strategy="S-TA"
)

ic_nta = estimate_all_term_ic(
    comparison_text,
    strategy="N-TA",
    N=N
)

comparison = ic_sta.merge(
    ic_nta,
    on=["index", "term"],
    suffixes=("_S-TA", "_N-TA")
)

display(comparison)

assert len(ic_sta) == len(ic_nta) == len(whitespace_terms(comparison_text))
assert np.isfinite(ic_sta["IC"]).all()
assert np.isfinite(ic_nta["IC"]).all()
assert (ic_sta["IC"] > 0).all()
assert (ic_nta["IC"] > 0).all()

tpi_sta = tpi_from_spans(
    ic_sta,
    term_indices(comparison_text, ["Geneva"])
)["TPI"]

tpi_nta = tpi_from_spans(
    ic_nta,
    term_indices(comparison_text, ["Geneva"])
)["TPI"]

print("TPI (S-TA):", tpi_sta)
print("TPI (N-TA):", tpi_nta)

assert 0 <= tpi_sta <= 1
assert 0 <= tpi_nta <= 1

print("✓ S-TA / N-TA structural consistency checks passed.")


,index,term,probability_S-TA,IC_S-TA,probability_N-TA,IC_N-TA
0,0,I,0.000433,7.744921,0.000300,8.112348
1,1,moved,0.198602,1.616453,0.295088,1.220481
2,2,to,0.994138,0.005880,0.994138,0.005880
3,3,Geneva,0.000788,7.145837,0.000788,7.145837
4,4,from,0.441273,0.818092,0.441273,0.818092
5,5,Brussels,0.001321,6.629283,0.001321,6.629283
6,6,in,0.879673,0.128205,0.924481,0.078523
7,7,2022,0.000002,12.930162,0.000003,12.741481


TPI (S-TA): 0.8069675310881399
TPI (N-TA): 0.8055656519168407
✓ S-TA / N-TA structural consistency checks passed.


## 11. Edge cases

We test cases that could otherwise silently break the implementation:

- empty text;
- punctuation-only text;
- a single-term text;
- multi-subtoken terms;
- punctuation attached to the end of a sentence.


In [61]:
# Empty text
empty_ic = estimate_all_term_ic("", strategy="N-TA", N=N)
assert empty_ic.empty

# Punctuation-only text
punct_ic = estimate_all_term_ic("!!! ... ???", strategy="N-TA", N=N)
assert punct_ic.empty

# Single term
single_ic = estimate_all_term_ic("Geneva", strategy="N-TA", N=N)
assert len(single_ic) == 1
assert np.isfinite(single_ic.loc[0, "IC"])

# Multi-subtoken / punctuation check
edge_text = "I visited Switzerland."
edge_terms = whitespace_terms(edge_text)
edge_ic = estimate_all_term_ic(edge_text, strategy="N-TA", N=N)

assert edge_terms[-1]["term"] == "Switzerland"
assert edge_ic["term"].tolist() == [x["term"] for x in edge_terms]
assert np.isfinite(edge_ic["IC"]).all()

print("✓ Edge-case tests passed.")


✓ Edge-case tests passed.


## 12. Final validation summary

The TPI implementation is considered structurally validated if all assertions
above pass.

The validation establishes:

- **Arithmetic correctness:** no affected terms gives TPI = 1; all affected
  terms gives TPI = 0.
- **Monotonic behaviour:** adding affected information cannot increase TPI.
- **Deletion support:** information can be counted even when no replacement
  exists.
- **Replacement/generalization support:** affected original spans are weighted
  by their IC.
- **Multiple-span support:** information loss accumulates across affected terms.
- **Token alignment:** punctuation does not prevent terms such as `2022` from
  being identified correctly.
- **S-TA/N-TA structural consistency:** both strategies produce valid
  term-level IC estimates.

### Interpretation

A successful validation does **not** establish that TPI is a perfect measure
of content preservation. It establishes that the implementation behaves as
intended on controlled examples.

For the interview anonymization experiments, the next step is to obtain
reliable affected spans from the anonymization process and then compute TPI
on the real original/anonymized utterance pairs.


## 13. References

- Manzanares-Salor, B., Sánchez, D., & Lison, P. (2026).
  *Unsupervised utility evaluation of text anonymization methods via neural
  language models*. Neural Networks.
- TAE (Text Anonymization Evaluation) framework.
